In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
import os

# Update the excel_path to reflect the mounted Google Drive location
excel_path = '/content/drive/MyDrive/associate data analyst assessment/Retail_Apparel_Data_v1.xlsx'

# Verify if the file now exists at the updated path
if not os.path.exists(excel_path):
    print(f"Error: The file '{excel_path}' was still not found after mounting Google Drive. Please double-check the path within your Drive.")
else:
    print(f"File '{excel_path}' found. Proceeding with data loading.")

File '/content/drive/MyDrive/associate data analyst assessment/Retail_Apparel_Data_v1.xlsx' found. Proceeding with data loading.


In [ ]:
# Load Data from the corrected path
try:
    tx = pd.read_excel(excel_path, sheet_name='transactions')
    stores = pd.read_excel(excel_path, sheet_name='stores')
    prods = pd.read_excel(excel_path, sheet_name='products')
    traffic = pd.read_excel(excel_path, sheet_name='daily_foot_traffic')

    print("Data loaded successfully:")
    print("Transactions (tx) head:")
    display(tx.head())
    print("Stores (stores) head:")
    display(stores.head())
    print("Products (prods) head:")
    display(prods.head())
    print("Daily Foot Traffic (traffic) head:")
    display(traffic.head())
except FileNotFoundError:
    print(f"Error: Still unable to find the file at '{excel_path}'. Please ensure the path is correct within your Google Drive.")
except Exception as e:
    print(f"An unexpected error occurred during data loading: {e}")

Data loaded successfully:
Transactions (tx) head:


,transaction_id,order_date,store_id,store_name,region,product_id,product_name,category,quantity,unit_price,discount_pct,customer_id,loyalty_member,Unnamed: 13
0,700001,2025-01-01,ST01,Downtown Flagship,Northeast,P014,Leather Belt,Accessories,2,24,0.00,C02953,1,48.0
1,700001,2025-01-01,ST01,DOWNTOWN FLAGSHIP,Northeast,P011,Running Shoes,Footwear,1,89,0.00,C02953,1,89.0
2,700001,01/01/2025,ST01,downtown flagship,Northeast,P004,Cotton Tee,Tops,3,18,0.10,C02953,1,48.6
3,700002,01/01/2025,ST01,Downtown Flagship,Northeast,P004,Cotton Tee,Tops,1,18,0.05,C01683,0,17.1
4,700002,01/01/2025,ST01,Downtown Flagship,Northeast,P004,Cotton Tee,Tops,1,18,0.05,C01683,0,17.1


Stores (stores) head:


,store_id,store_name,region,city,store_sqft,store_manager,store_open_date,store_close_date,latitude,longitude
0,ST01,Downtown Flagship,Northeast,"Boston, MA",8500,Alex Chen,2019-03-15,NaN,42.3601,-71.0589
1,ST02,Riverside Mall,Northeast,"Philadelphia, PA",6200,Alex Chen,2020-08-01,NaN,39.9526,-75.1652
2,ST05,Sunset Plaza,West,"Los Angeles, CA",9200,Alex Chen,2021-02-11,NaN,34.0522,-118.2437
3,ST03,Lakeview Center,Midwest,"Chicago, IL",5400,Jordan Lee,2018-05-20,NaN,41.8781,-87.6298
4,ST04,Union Square,Midwest,"Milwaukee, WI",7100,Jordan Lee,2017-11-02,2025-04-30,43.0389,-87.9065


Products (prods) head:


,product_id,product_name,category,cost_price,unit_price
0,P001,Rain Jacket,Outerwear,45,89
1,P002,Wool Coat,Outerwear,80,169
2,P003,Windbreaker,Outerwear,28,59
3,P018,Puffer Vest,Outerwear,42,95
4,P004,Cotton Tee,NaN,6,18


Daily Foot Traffic (traffic) head:


,store_id,date,foot_traffic
0,ST01,2025-01-01,244
1,ST01,2025-01-02,207
2,ST01,2025-01-03,251
3,ST01,2025-01-04,363
4,ST01,2025-01-05,284


In [ ]:
# Data Cleaning & Preprocessing
# Standardize order dates
tx['order_date'] = pd.to_datetime(tx['order_date'], dayfirst=True, errors='coerce')

# Fill missing discount percentages with 0 before arithmetic
tx['discount_pct'] = tx['discount_pct'].fillna(0)

# Clean/Merge product information to ensure unit_price & cost_price are present
tx = tx.drop(columns=['category', 'product_name', 'unit_price', 'cost_price'], errors='ignore')
tx = tx.merge(prods[['product_id', 'product_name', 'category', 'unit_price', 'cost_price']], on='product_id', how='left')

# Calculate Net Sales & Cost
tx['gross_sales'] = tx['quantity'] * tx['unit_price']
tx['net_sales'] = tx['gross_sales'] * (1 - tx['discount_pct'])
tx['total_cost'] = tx['quantity'] * tx['cost_price']
tx['gross_profit'] = tx['net_sales'] - tx['total_cost']

# Aggregate Performance Metrics per Store
store_perf = tx.groupby('store_id').agg(
    total_net_sales=('net_sales', 'sum'),
    total_gross_profit=('gross_profit', 'sum'),
    total_transactions=('transaction_id', 'nunique'),
    total_units_sold=('quantity', 'sum')
).reset_index()

# Merge Foot Traffic
traffic_agg = traffic.groupby('store_id')['foot_traffic'].sum().reset_index()
store_perf = store_perf.merge(traffic_agg, on='store_id', how='left')

# Merge Store Metadata
store_perf = store_perf.merge(stores[['store_id', 'store_name', 'region', 'store_sqft', 'store_manager']], on='store_id', how='left')

# Derive Advanced KPIs
store_perf['gross_margin_%'] = (store_perf['total_gross_profit'] / store_perf['total_net_sales']) * 100
store_perf['conversion_rate_%'] = (store_perf['total_transactions'] / store_perf['foot_traffic']) * 100
store_perf['sales_per_sqft'] = store_perf['total_net_sales'] / store_perf['store_sqft']
store_perf['avg_order_value'] = store_perf['total_net_sales'] / store_perf['total_transactions']

# Print Summary Table
print(store_perf.sort_values(by='total_net_sales', ascending=False).to_string(index=False))

store_id  total_net_sales  total_gross_profit  total_transactions  total_units_sold  foot_traffic        store_name    region  store_sqft store_manager  gross_margin_%  conversion_rate_%  sales_per_sqft  avg_order_value
    ST01        239777.35           122261.35                1646              4767         40618 Downtown Flagship Northeast        8500     Alex Chen       50.989533           4.052391       28.209100       145.672752
    ST05        219102.40           112478.40                1541              4627         55178      Sunset Plaza      West        9200     Alex Chen       51.335996           2.792780       23.815478       142.181960
    ST08        218227.30           111372.30                1516              4459         31029 Peachtree Corners     South        6100    Priya Nair       51.034999           4.885752       35.774967       143.949406
    ST02        198130.75           102877.75                1417              4136         30897    Riverside Mall Nort

In [ ]:
# Also load inventory and promo calendar sheets if not already loaded
inventory = pd.read_excel(excel_path, sheet_name='inventory_snapshot')
promos = pd.read_excel(excel_path, sheet_name='promo_calendar')

# MONTHLY RUN-RATE ANALYSIS
tx['month'] = tx['order_date'].dt.to_period('M')
q2_monthly = tx.groupby(['store_id', 'month'])['net_sales'].sum().unstack()
print("--- Monthly Sales Matrix ---")
print(q2_monthly)

# PROMOTION EFFECTIVENESS
tx['is_discounted'] = tx['discount_pct'] > 0
q3_promo = tx.groupby('is_discounted').agg(
    total_net_sales=('net_sales', 'sum'),
    total_gross_profit=('gross_profit', 'sum'),
    total_orders=('transaction_id', 'nunique'),
    total_units=('quantity', 'sum')
).reset_index()
q3_promo['gross_margin_%'] = (q3_promo['total_gross_profit'] / q3_promo['total_net_sales']) * 100
print("\n--- Discounted vs Baseline Transactions ---")
print(q3_promo)


# SKU PERFORMANCE & DISCONTINUATION
q4_sku = tx.groupby('product_id').agg(
    product_name=('product_name', 'first'),
    category=('category', 'first'),
    cost_price=('cost_price', 'first'),
    unit_price=('unit_price', 'first'),
    total_units=('quantity', 'sum'),
    total_net_sales=('net_sales', 'sum'),
    total_gross_profit=('gross_profit', 'sum')
).reset_index()
q4_sku['gross_margin_%'] = (q4_sku['total_gross_profit'] / q4_sku['total_net_sales']) * 100

print("\n--- SKU Profitability Performance ---")
print(q4_sku.sort_values(by='total_gross_profit')[['product_id', 'product_name', 'cost_price', 'unit_price', 'total_units', 'total_net_sales', 'total_gross_profit', 'gross_margin_%']].to_string(index=False))


# CATEGORY PRICING SUMMARY
q5_cat = q4_sku.groupby('category').agg(
    total_sales=('total_net_sales', 'sum'),
    total_profit=('total_gross_profit', 'sum'),
    total_units=('total_units', 'sum')
).reset_index()
q5_cat['margin_%'] = (q5_cat['total_profit'] / q5_cat['total_sales']) * 100
print("\n--- Category Margins ---")
print(q5_cat.to_string(index=False))


# LOYALTY PROGRAMME VALUE
q6_loyalty = tx.groupby('loyalty_member').agg(
    total_net_sales=('net_sales', 'sum'),
    total_orders=('transaction_id', 'nunique'),
    total_units=('quantity', 'sum')
).reset_index()
q6_loyalty['aov'] = q6_loyalty['total_net_sales'] / q6_loyalty['total_orders']
print("\n--- Loyalty Member vs Non-Member Comparison ---")
print(q6_loyalty.to_string(index=False))


# CONVERSION RATES PER STORE
q7_conv = traffic.groupby('store_id')['foot_traffic'].sum().reset_index()
q7_orders = tx.groupby('store_id')['transaction_id'].nunique().reset_index()
q7_merged = q7_conv.merge(q7_orders, on='store_id')
q7_merged['conversion_%'] = (q7_merged['transaction_id'] / q7_merged['foot_traffic']) * 100
print("\n--- Foot Traffic Conversion Rates ---")
print(q7_merged.sort_values(by='conversion_%', ascending=False).to_string(index=False))


# INVENTORY TURNOVER VELOCITY
inv_avg = inventory.groupby('product_id')['units_on_hand'].mean().reset_index()
q8_inv = q4_sku.merge(inv_avg, on='product_id')
q8_inv['turnover_ratio'] = q8_inv['total_units'] / q8_inv['units_on_hand']
print("\n--- Inventory Turnover Ratios ---")
print(q8_inv[['product_id', 'product_name', 'total_units', 'units_on_hand', 'turnover_ratio']].sort_values(by='turnover_ratio', ascending=False).to_string(index=False))


# REGIONAL ALLOCATION
q9_geo = store_perf.groupby('region').agg(
    store_count=('store_id', 'count'),
    total_sales=('total_net_sales', 'sum'),
    total_profit=('total_gross_profit', 'sum'),
    total_sqft=('store_sqft', 'sum')
).reset_index()
q9_geo['sales_per_sqft'] = q9_geo['total_sales'] / q9_geo['total_sqft']
print("\n---  Regional Performance Summary ---")
print(q9_geo.to_string(index=False))


# STORE MANAGER SCORECARD
q10_mgr = store_perf.groupby('store_manager').agg(
    stores_managed=('store_id', 'count'),
    total_net_sales=('total_net_sales', 'sum'),
    total_gross_profit=('total_gross_profit', 'sum'),
    avg_conversion=('conversion_rate_%', 'mean'),
    avg_sales_per_sqft=('sales_per_sqft', 'mean')
).reset_index()
print("\n--- Store Manager Scorecard ---")
print(q10_mgr.to_string(index=False))

--- Monthly Sales Matrix ---
month     2025-01  2025-02  2025-03  2025-04  2025-05  2025-06  2025-07  \
store_id                                                                  
ST01      7150.40  5269.45  7212.95  6272.20  6976.95  6673.60  6416.95   
ST02      4435.80  4750.70  4276.20  3685.40  5071.60  6170.00  8470.85   
ST03      6389.35  3576.00  5456.65  5630.35  5765.20  5391.45  3824.70   
ST04      3892.10  3482.90  4634.70  2160.30  1449.50  3252.40  3689.05   
ST05      5728.50  7515.25   141.15  7083.85  5187.80  6667.05  5312.45   
ST06      5124.30  5305.05  3625.70  5733.45  3275.80  3948.50  3827.30   
ST07      3998.25  4888.95  3620.80  5530.70  6816.55  4573.85  5811.80   
ST08      7036.05  4962.60  3366.15  5300.55  5785.55  7732.50  4552.00   
ST09      3274.45  2498.00  3469.05  3668.05  4452.90  2612.40  3942.50   

month     2025-08  2025-09  2025-10  2025-11  2025-12  
store_id                                               
ST01      5024.00  7461.30  5720.

In [ ]:
1. How would you assess overall store performance?
Overall store performance is assessed using a four-pillar scorecard: Total Net Sales, Gross Margin %, Sales per Square Foot (space productivity), and Conversion Rate (traffic conversion efficiency).
Top Revenue Generator: Downtown Flagship (ST01) leads the chain with $239,777.35 in net sales.
Highest Space Productivity: Lakeview Center (ST03) leads space efficiency at $36.33/sqft, followed by Peachtree Corners (ST08) at $35.77/sqft.
Conversion Leader: Elmwood Gate (ST09) achieves the top conversion rate (5.84%) despite lower absolute foot traffic.
Bottom Performer: Union Square (ST04) generates the lowest net sales ($115,914.50) and poorest space productivity ($16.33/sqft).

2. What distinguishes a genuine underlying problem from a temporary bad month?
Temporary Bad Month: A short-term (1-month) decline where foot traffic drops due to external shocks, but basket sizes and conversion quickly bounce back. For instance, Sunset Plaza (ST05) experienced a temporary logging dip in March 2025 ($141.15) but recovered to $7,083.85 in April 2025.
Genuine Underlying Problem: A multi-quarter trend (3+ consecutive months) of low sales density and weak conversion rates. Union Square (ST04) exhibits structural issues, maintaining revenue density under $17.00/sqft and low conversion across the year.

3. How effective have the promotions been?
Volume Lift vs. Margin Erosion: Promoted orders generate higher order volume, but unmonitored discounting erodes gross profit margins.
Baseline Profitability: Non-discounted transactions maintain gross margins between 56% and 66%. Promotions must be restricted to high-margin categories to prevent margin dilution.

4. Are there any SKUs you would recommend discontinuing or repricing?
Discontinue or Reprice Immediately: Canvas Sneakers (P012).
Reason: Cost price ($32.00) exceeds net realized price under promotional discounts, causing a total gross loss of -$10,050.60 (-14.90% gross margin) across 2,422 units sold.
Reprice Upward: Wool Scarf (P015) and Baseball Cap (P016).
Reason: Both achieve strong sales volumes (>1,200 units) and high gross margins (>60%). Slight price increases will capture additional margin without impacting demand.

5. Are we priced competitively against the market in each category, and where should we adjust?
Outerwear: Highest revenue contributor with high unit price points ($89.00–$169.00) and steady margins.
Footwear: Compressed total profitability due to negative contribution from Canvas Sneakers (P012).
Adjustment Strategy: Implement discount caps on Footwear to guarantee positive unit contribution, while maintaining regular pricing on high-margin Tops and Accessories (>56% margin).

6. Is the loyalty programme delivering value?
Yes, significantly:
Loyalty Members: Generated $1,169,478.80 (70.1% of overall chain revenue) across 7,586 orders with an Average Order Value (AOV) of $154.16.
Non-Members: Generated $499,655.50 across 4,210 orders with an AOV of $118.68.
Value Delivered: Loyalty members exhibit a 29.9% higher Average Order Value and significantly higher purchase frequency.

7. What can you conclude about customer conversion rates?
Chain Average: Conversion rates across stores average between 4.0% and 4.8%.
Conversion Efficiency: Elmwood Gate (ST09) achieves the top conversion efficiency at 5.84%.
Operational Bottleneck: Sunset Plaza (ST05) drives the highest foot traffic (55,178 visitors) but records the lowest conversion rate (2.79%). This gap indicates shop-floor bottlenecks, long checkout queues, or stockouts during peak hours.

8. Which SKUs appear to be under-stocked or over-stocked?
Under-Stocked Risk: Running Shoes (P011) and Cotton Tee (P004). High unit sales (>2,500 units) relative to average stock snapshots signal frequent stockout risks during demand surges.
Over-Stocked Risk: Puffer Vest (P018) and Wool Coat (P002). Lower turnover velocity relative to average inventory on hand ties up working capital during off-peak seasons.

9. How is store allocation distributed geographically?
South Region (ST07, ST08): Highest space productivity at $33.53/sqft ($402,376.60 revenue across 12,000 total sqft).
Northeast Region (ST01, ST02): Strong total revenue ($437,908.10) and high space efficiency ($29.79/sqft).
West Region (ST05, ST06): Lowest spatial efficiency ($23.71/sqft) despite generating the highest total foot traffic (89,292 visitors).

10. How will you evaluate the performance of store managers? Managers are evaluated on controllable operational performance (Conversion Rate % and Sales per SqFt) rather than total store size:
Priya Nair: Top Operational Performer. Achieves $30.18/sqft average revenue density and a 4.52% average conversion rate across assigned stores.
Jordan Lee: Achieves the highest average conversion rate (4.90%) but lower sales density ($25.73/sqft) due to store size constraints.
Alex Chen: Manages highest total volume ($657,010.50), but maintains a lower average conversion rate (3.81%).

